# models

> Data models and URL bundles for the alignment package

In [ ]:
#| default_exp models

In [ ]:
#| export
from typing import Optional, List, Dict, Any
from typing_extensions import TypedDict
from dataclasses import dataclass, field, asdict

from cjm_fasthtml_card_stack.core.models import CardStackUrls

## AlignmentStepState

TypedDict for Phase 2 right column (VAD alignment) state.

In [ ]:
#| export
class AlignmentStepState(TypedDict, total=False):
    """State for Phase 2 (right column): Temporal Alignment."""

    # --- Workflow-specific ---
    is_initialized: bool  # Whether VAD data has been fetched
    vad_chunks: List[Dict[str, Any]]  # VAD chunks (serialized VADChunk)
    media_path: Optional[str]  # Path to first audio file (backward compat)
    media_paths: List[str]  # Ordered list of audio file paths
    audio_duration: Optional[float]  # Total audio duration in seconds

    # --- Card stack view state ---
    focused_chunk_index: int  # Currently focused VAD chunk (default 0)
    visible_count: int  # Number of visible cards in viewport
    is_auto_mode: bool  # Whether card count is in auto-adjust mode
    card_width: int  # Card stack width in rem units

    # --- Audio playback ---
    auto_navigate: bool  # Auto-advance to next chunk when audio finishes
    playback_speed: float  # Pitch-preserving playback speed (1.0 = normal)

## VADChunk

Represents a voice activity detection time range from the VAD plugin.

In [ ]:
#| export
@dataclass
class VADChunk:
    """A voice activity detection time range."""
    
    index: int  # Chunk index in sequence (global across all audio files)
    start_time: float  # Start time in seconds (relative to its audio file)
    end_time: float  # End time in seconds (relative to its audio file)
    audio_file_index: int = 0  # Which audio file this chunk belongs to
    
    @property
    def duration(self) -> float:  # Duration in seconds
        """Calculate chunk duration."""
        return self.end_time - self.start_time
    
    def to_dict(self) -> Dict[str, Any]:  # Dictionary representation
        """Convert to dictionary for JSON serialization."""
        return asdict(self)
    
    @classmethod
    def from_dict(
        cls,
        data: Dict[str, Any]  # Dictionary representation
    ) -> "VADChunk":  # Reconstructed VADChunk
        """Create from dictionary."""
        data = data.copy()
        # Remove legacy fields if present
        data.pop("assigned_segment", None)
        # Default audio_file_index for backward compat
        if "audio_file_index" not in data:
            data["audio_file_index"] = 0
        return cls(**data)

## AlignmentUrls

URL bundle for Phase 2 alignment route handlers and renderers.

In [ ]:
#| export
@dataclass
class AlignmentUrls:
    """URL bundle for Phase 2 alignment route handlers and renderers."""

    # Card stack navigation and viewport (from cjm-fasthtml-card-stack library)
    card_stack: CardStackUrls = field(default_factory=CardStackUrls)

    # Workflow-specific: initialization
    init: str = ""  # Initialize alignment (fetch VAD data)

    # Audio serving
    audio_src: str = ""  # Audio file serving URL base

    # Audio controls
    toggle_auto_nav: str = ""  # Toggle auto-navigate on/off
    speed_change: str = ""  # Persist playback speed changes

## Tests

In [ ]:
# VADChunk with audio_file_index
chunk = VADChunk(index=0, start_time=1.0, end_time=3.5, audio_file_index=2)
assert chunk.audio_file_index == 2
assert chunk.duration == 2.5
d = chunk.to_dict()
assert d["audio_file_index"] == 2
assert VADChunk.from_dict(d).audio_file_index == 2

# Backward compat: missing audio_file_index defaults to 0
legacy = {"index": 5, "start_time": 0.0, "end_time": 1.0}
assert VADChunk.from_dict(legacy).audio_file_index == 0

# Legacy field removal still works
legacy_with_assigned = {"index": 0, "start_time": 0.0, "end_time": 1.0, "assigned_segment": None}
assert VADChunk.from_dict(legacy_with_assigned).audio_file_index == 0
print("VADChunk model tests passed")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()